In [49]:
import os
import pandas as pd
import numpy as np
from pydbmanager.connection import DatabaseConnection
from pydbmanager.operations import DatabaseOperations
from dotenv import load_dotenv
import warnings
import json
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
# Load environment variables
load_dotenv()

# Initialize and test database connection
db = DatabaseConnection()
# Initialize database operations (this also creates and stores the connection)
db_ops = DatabaseOperations()

# Check connection status
if db_ops.conn:
    print("\u2705 Connection Successful!")
    db_ops.close()
else:
    print("\u274c Connection Failed!")

2025-05-11 04:11:35,386 - INFO - Database connection established successfully.
2025-05-11 04:11:35,386 - INFO - Database connection closed


✅ Connection Successful!


## 2. Incident Registration data

In [3]:
with open('../../sql/tbi_incident_registration_data.sql', 'r') as file:
    tbi_incident_reg_sql = file.read()

In [4]:
tbi_incident_reg = db_ops.query_data(tbi_incident_reg_sql)
tbi_incident_reg.head()

2025-05-11 04:11:39,186 - WARNING - Database connection lost. Reconnecting...
2025-05-11 04:11:39,188 - INFO - Database connection established successfully.
2025-05-11 04:11:39,275 - INFO - Data fetched in 0.0872 seconds


Data fetched successfully!
Dataframe Size (954, 8)


,patient_id,tbi_incident_date,injury_from,head_hit_location,num_head_hit_location,total_tbi,immediate_symptoms_resulting,describe_event
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-02,Accident,Top of Head,1,1,"Confusion, Dazed or Vacant Stare, Disorientati...","Automobile accident, ran into cars and flipped..."
1,00469456-99a5-4c99-aa48-a918986c7c45,2021-02-09,Subarachnoid haemorrhage,Right Side of Head,1,1,"Confusion, Dazed or Vacant Stare, Disorientati...",I collapsed at work showing stroke like sympto...
2,012bfc75-70c1-4007-ab9e-1f4ee45bd537,2014-08-12,Accident,Entire head. texture trailer from semi that hi...,1,2,"Loss of Consciousness,Disorientation,Incoheren...",Hit by a car checking my mail. The car hit me ...
3,012eccde-f0b9-4ae2-aebf-3c821c461545,2022-10-21,Fall,Left Side of Head,1,1,"Incoherent Speech,Disorientation,Confusion,Daz...",I fell down stairs
4,01786d46-2829-42e2-8d50-135b7e212ea3,2019-03-01,Accident,Back of Head,1,3,"Disorientation,Confusion,Extreme pain in the f...",Not sure which day in March. I went to sit bac...


In [8]:
# tbi_incident_reg.to_clipboard()

In [5]:
tbi_incident_reg.isna().sum()

patient_id                       0
tbi_incident_date                0
injury_from                      7
head_hit_location               21
num_head_hit_location            0
total_tbi                        0
immediate_symptoms_resulting    10
describe_event                   0
dtype: int64

In [6]:
## total tbi we will cap at 30
## num head hit location will put zero for na values
## injury_from we will put not known for na values
## immediate_symptioms_resulting we will put None for na values

In [7]:
tbi_incident_reg.dtypes

patient_id                      object
tbi_incident_date               object
injury_from                     object
head_hit_location               object
num_head_hit_location           object
total_tbi                        int64
immediate_symptoms_resulting    object
describe_event                  object
dtype: object

In [8]:
tbi_incident_reg['total_tbi'] = np.where(tbi_incident_reg['total_tbi'] <= 30, tbi_incident_reg['total_tbi'], 30)
tbi_incident_reg['num_head_hit_location'] = tbi_incident_reg['num_head_hit_location'].str.lower().replace('null', 0)
tbi_incident_reg['num_head_hit_location'] = tbi_incident_reg['num_head_hit_location'].astype(int).fillna(0).astype(int)
tbi_incident_reg['injury_from'] = tbi_incident_reg['injury_from'].fillna('Not Known')
tbi_incident_reg['immediate_symptoms_resulting'] = tbi_incident_reg['immediate_symptoms_resulting'].fillna('None')
tbi_incident_reg['head_hit_location'] = tbi_incident_reg['head_hit_location'].fillna('Not known')

In [9]:
tbi_incident_reg['tbi_incident_date'] = pd.to_datetime(tbi_incident_reg['tbi_incident_date'],errors='coerce')

In [10]:
max(tbi_incident_reg['tbi_incident_date']), min(tbi_incident_reg['tbi_incident_date'])

(Timestamp('2025-01-29 00:00:00'), Timestamp('1901-01-01 00:00:00'))

In [11]:
tbi_incident_reg.isna().sum()

patient_id                      0
tbi_incident_date               1
injury_from                     0
head_hit_location               0
num_head_hit_location           0
total_tbi                       0
immediate_symptoms_resulting    0
describe_event                  0
dtype: int64

In [12]:
tbi_incident_reg.shape

(954, 8)

## 3. Worst Top 3 symptoms

In [13]:
with open('../../sql/worst_3_symptoms.sql') as file:
    worst_3_symptoms_sql = file.read()

In [14]:
worst_symptoms = db_ops.query_data(worst_3_symptoms_sql)
worst_symptoms.head()

2025-05-11 04:15:07,508 - INFO - Data fetched in 0.0541 seconds


Data fetched successfully!
Dataframe Size (1192, 6)


,patient_id,symptom_id,id,category,subcategory,factor
0,00e179cd-5edb-47fe-be53-b1e96e905433,7,7,medical,cognitive,"Brain Fog, Lack of Focus"
1,00e179cd-5edb-47fe-be53-b1e96e905433,8,8,medical,cognitive,Short Term Memory Loss
2,00e179cd-5edb-47fe-be53-b1e96e905433,10,10,medical,cognitive,Slow Thinking or Processing
3,012eccde-f0b9-4ae2-aebf-3c821c461545,7,7,medical,cognitive,"Brain Fog, Lack of Focus"
4,012eccde-f0b9-4ae2-aebf-3c821c461545,8,8,medical,cognitive,Short Term Memory Loss


In [15]:
worst_symptoms.shape

(1192, 6)

In [16]:
worst_symptoms.isna().sum()

patient_id     0
symptom_id     0
id             0
category       0
subcategory    0
factor         0
dtype: int64

In [17]:
# worst_symptoms.to_clipboard()

## 4. ALL RECORDED MEDICAL SYMPTOMS

In [18]:
with open('../../sql/recorded_medical_symptoms.sql') as file:
    medical_symptoms_sql = file.read()

In [19]:
med_sympt = db_ops.query_data(medical_symptoms_sql)
med_sympt.head()

2025-05-11 04:15:15,358 - INFO - Data fetched in 0.1715 seconds


Data fetched successfully!
Dataframe Size (6986, 7)


,patient_id,symptom_id,id,category,subcategory,factor,prime
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,26,26,medical,physical,Lack of Coordination,true
1,0006ad41-c2d3-4994-8aab-7a3a107d50aa,511,511,medical,speech,Limited social engagement,true
2,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1009,1009,medical,cognitive,Can't sleep or relax,NULL
3,00469456-99a5-4c99-aa48-a918986c7c45,3,3,medical,sleep,Fatigue,true
4,00469456-99a5-4c99-aa48-a918986c7c45,8,8,medical,cognitive,Short Term Memory Loss,true


In [20]:
med_sympt.shape

(6986, 7)

In [21]:
med_sympt.isna().sum()

patient_id     0
symptom_id     0
id             0
category       0
subcategory    0
factor         0
prime          0
dtype: int64

In [22]:
def build_json(group):
    category = group['category'].iloc[0]  
    symptoms = group[['subcategory', 'factor', 'prime']].to_dict(orient='records')
    return {'category': category, 'symptoms': symptoms}

In [23]:
med_sympt_result = med_sympt.groupby('patient_id').apply(build_json).reset_index(name='symptom_json')

In [24]:
med_sympt_result.shape

(983, 2)

In [25]:
med_sympt_result.head()

,patient_id,symptom_json
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,"{'category': 'medical', 'symptoms': [{'subcate..."
1,00469456-99a5-4c99-aa48-a918986c7c45,"{'category': 'medical', 'symptoms': [{'subcate..."
2,00e179cd-5edb-47fe-be53-b1e96e905433,"{'category': 'medical', 'symptoms': [{'subcate..."
3,012bfc75-70c1-4007-ab9e-1f4ee45bd537,"{'category': 'medical', 'symptoms': [{'subcate..."
4,012eccde-f0b9-4ae2-aebf-3c821c461545,"{'category': 'medical', 'symptoms': [{'subcate..."


In [26]:
# med_sympt_result.to_clipboard()

## 5. ALL SDOH RECORDS  AT THE TIME OF REGISTRATION

In [27]:
with open('../../sql/SDOH_data.sql') as file:
    sdoh_sql = file.read()

In [28]:
sdoh_df = db_ops.query_data(sdoh_sql)
sdoh_df.head()

2025-05-11 04:15:24,332 - INFO - Data fetched in 0.1362 seconds


Data fetched successfully!
Dataframe Size (5219, 5)


,patient_id,symptom_id,category,subcategory,factor
0,00e179cd-5edb-47fe-be53-b1e96e905433,52,SDOH,wellness,Exercise
1,00e179cd-5edb-47fe-be53-b1e96e905433,64,SDOH,wellness,Stress
2,00e179cd-5edb-47fe-be53-b1e96e905433,1136,SDOH,wellness,Muscle Pain
3,012eccde-f0b9-4ae2-aebf-3c821c461545,32,SDOH,travel,Car
4,012eccde-f0b9-4ae2-aebf-3c821c461545,50,SDOH,wellness,Dehydration


In [29]:
sdoh_df.shape

(5219, 5)

In [30]:
# sdoh_df.to_clipboard()

In [31]:
def build_sdoh_json(group):
    category = group['category'].iloc[0]
    subcat_dict = (
        group.groupby('subcategory')['factor']
        .apply(lambda x: sorted(set(x.dropna())))
        .to_dict()
    )
    return {'category': category, 'subcategories': subcat_dict}

In [32]:
sdoh_df_result = sdoh_df.groupby('patient_id').apply(build_sdoh_json).reset_index(name='sdoh_json')

In [33]:
sdoh_df_result.shape

(409, 2)

In [34]:
sdoh_df_result.head()

,patient_id,sdoh_json
0,00e179cd-5edb-47fe-be53-b1e96e905433,"{'category': 'SDOH', 'subcategories': {'wellne..."
1,012eccde-f0b9-4ae2-aebf-3c821c461545,"{'category': 'SDOH', 'subcategories': {'travel..."
2,021a5e27-8029-4f0c-b268-cee04dbc40b9,"{'category': 'SDOH', 'subcategories': {'Dietar..."
3,022242f1-bcae-4230-9b5d-e03b1e3e25fa,"{'category': 'SDOH', 'subcategories': {'wellne..."
4,02b72dc4-3645-4f19-be5a-989e9147544e,"{'category': 'SDOH', 'subcategories': {'enviro..."


In [35]:
# sdoh_df_result.to_clipboard()

## 6. ALL THERAPIES AT THE TIME OF REGISTRATION

In [36]:
with open('../../sql/therapies_at_reg.sql') as file:
    therapies_sql = file.read() 

In [37]:
therap_df = db_ops.query_data(therapies_sql)
therap_df.head()

2025-05-11 04:15:29,970 - INFO - Data fetched in 0.0309 seconds


Data fetched successfully!
Dataframe Size (1866, 4)


,patient_id,therapies_id,therapies,category
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,11,Acupuncture,NULL
1,0006ad41-c2d3-4994-8aab-7a3a107d50aa,12,Chiropractic / Functional Neurology,NULL
2,0006ad41-c2d3-4994-8aab-7a3a107d50aa,13,Massage Therapy,NULL
3,0006ad41-c2d3-4994-8aab-7a3a107d50aa,14,Pain Management,NULL
4,0006ad41-c2d3-4994-8aab-7a3a107d50aa,15,Physical Therapy,NULL


In [38]:
# therap_df.to_clipboard()

In [39]:
def build_therapy_json(group):
    therapy_dict = (
        group.groupby('category')['therapies']
        .apply(lambda x: sorted(set(x.dropna())))
        .to_dict()
    )
    return therapy_dict

In [40]:
therap_df_result = therap_df.groupby('patient_id').apply(build_therapy_json).reset_index(name='therapy_json')
therap_df_result.shape

(399, 2)

In [41]:
therap_df_result.head()

,patient_id,therapy_json
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,"{'NULL': ['Acupuncture', 'Chiropractic / Funct..."
1,00e179cd-5edb-47fe-be53-b1e96e905433,{'NULL': ['None']}
2,021a5e27-8029-4f0c-b268-cee04dbc40b9,"{'Alternative': ['Pain Management'], 'NULL': [..."
3,022242f1-bcae-4230-9b5d-e03b1e3e25fa,"{'Applied': ['Physical Therapy'], 'Chiropracti..."
4,02b72dc4-3645-4f19-be5a-989e9147544e,"{'Applied': ['Occupational Therapy', 'Physical..."


In [42]:
# therap_df_result.to_clipboard()

## 7. SYMPTOM LOGS OVER TIME

In [43]:
with open('../../sql/symptom_logs.sql') as file:
    symptom_logs_sql = file.read()  

In [44]:
symptom_logs = db_ops.query_data(symptom_logs_sql)
symptom_logs.head()

2025-05-11 04:15:40,560 - INFO - Data fetched in 6.0511 seconds


Data fetched successfully!
Dataframe Size (187218, 8)


,patient_about_id,symptom_date,logged_at,severity,category,subcategory,had_symptom,factor
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.665091,NULL,medical,cognitive,true,"Brain Fog, Lack of Focus"
1,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,emotional,true,Anxiety
2,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,emotional,true,Depression
3,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,emotional,true,No Motivation
4,0006ad41-c2d3-4994-8aab-7a3a107d50aa,1999-07-01 00:00:00,2022-06-08 16:49:20.774459,NULL,medical,sleep,true,Constipation


### Testing Updated PyDBManager package

In [45]:
therap_df_result.shape, therap_df_result.dtypes

((399, 2),
 patient_id      object
 therapy_json    object
 dtype: object)

In [46]:
therap_df_result.head()

,patient_id,therapy_json
0,0006ad41-c2d3-4994-8aab-7a3a107d50aa,"{'NULL': ['Acupuncture', 'Chiropractic / Funct..."
1,00e179cd-5edb-47fe-be53-b1e96e905433,{'NULL': ['None']}
2,021a5e27-8029-4f0c-b268-cee04dbc40b9,"{'Alternative': ['Pain Management'], 'NULL': [..."
3,022242f1-bcae-4230-9b5d-e03b1e3e25fa,"{'Applied': ['Physical Therapy'], 'Chiropracti..."
4,02b72dc4-3645-4f19-be5a-989e9147544e,"{'Applied': ['Occupational Therapy', 'Physical..."


In [48]:
create_table_sql = """
IF NOT EXISTS (
    SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = 'patient_symptoms'
)
BEGIN
    CREATE TABLE patient_symptoms (
        patient_id VARCHAR(100) PRIMARY KEY,
        therapy_json NVARCHAR(MAX)
    );
END
"""
db_ops.create_table(create_table_sql)

2025-05-11 04:18:06,112 - INFO - Query executed successfully!


True

In [50]:
therap_df_result['therapy_json'] = therap_df_result['therapy_json'].apply(lambda x: json.dumps(x))

# Now insert
success = db_ops.insert_dataframe(therap_df_result, table_name='patient_symptoms')


2025-05-11 04:18:38,666 - INFO - Inserted DataFrame into table 'patient_symptoms' successfully!
